In [20]:
#say no to warnings!
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = '3'
import tensorflow as tf
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

# Sentiment Analysis with Recursive Neural Networks

Let's try and resolve a task of Sentiment Analysis training, evaluating and comparing different architectures of Recursive Neural Networks.

## Dataset

We are going to be using a dataset from kaggle composed of 50k movie reviews each one labeled either ```positive``` or ```negative```.

## Architectures

We will be comparing the following RNNs architecture, trained and evaluated on the same dataset:
- Vanilla RNN
- Long Short-Term Memory RNN (LSTM)
- Gated Recurrent Unit RNN
- Bidirectional RNN 

## Dowload the dataset

In [ ]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

In [ ]:
import shutil
shutil.move(path, './my_datasets/imdb')

In [42]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, SimpleRNN, LSTM, GRU, Bidirectional
import pandas as pd

### Data preprocessing

In [23]:
df = pd.read_csv("my_datasets/imdb/IMDB Dataset.csv")

X = df.review.values
y = df.sentiment.values

In [24]:
le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, random_state=1)

In [25]:
tokenizer = Tokenizer(num_words=10000)

In [26]:
tokenizer.fit_on_texts(X_train)

train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences = tokenizer.texts_to_sequences(X_test)

vocabulary_size = len(tokenizer.index_word)+1

maxlen = len(max(train_sequences, key=len))

In [28]:
padded_train_sequences = pad_sequences(train_sequences, maxlen=maxlen)
padded_test_sequences = pad_sequences(test_sequences, maxlen=maxlen)

## Vanilla RNN

In [37]:
model = Sequential()
model.add(Embedding(input_dim=vocabulary_size, output_dim=128, input_length=maxlen))
model.add(SimpleRNN(64))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=["accuracy"]) # rmsprop suggested from documentation to avoid vanishing gradient
model.fit(padded_train_sequences, y_train, validation_split=.2, epochs=5, batch_size=256)

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 169s 2s/step - accuracy: 0.5989 - loss: 0.6551 - val_accuracy: 0.7543 - val_loss: 0.5230
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 178s 2s/step - accuracy: 0.7866 - loss: 0.4696 - val_accuracy: 0.7447 - val_loss: 0.5060
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 170s 2s/step - accuracy: 0.8392 - loss: 0.3739 - val_accuracy: 0.8371 - val_loss: 0.4096
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 176s 2s/step - accuracy: 0.8800 - loss: 0.2938 - val_accuracy: 0.8479 - val_loss: 0.3718
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 185s 2s/step - accuracy: 0.9149 - loss: 0.2211 - val_accuracy: 0.8120 - val_loss: 0.4427


In [39]:
model.evaluate(padded_test_sequences, y_test)

469/469 ━━━━━━━━━━━━━━━━━━━━ 25s 53ms/step - accuracy: 0.8144 - loss: 0.4454


[0.4453975260257721, 0.8144000172615051]

## Long Short-Term Memory RNN

In [41]:
lstm_model = Sequential()
lstm_model.add(Embedding(input_dim=vocabulary_size, output_dim=128, input_length=maxlen))
lstm_model.add(LSTM(64, activation='tanh'))
lstm_model.add(Dense(1, activation='sigmoid'))
lstm_model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.fit(padded_train_sequences, y_train, validation_split=.2, epochs=5, batch_size=256)

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 353s 3s/step - accuracy: 0.6650 - loss: 0.6066 - val_accuracy: 0.7889 - val_loss: 0.4746
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 376s 3s/step - accuracy: 0.8012 - loss: 0.4380 - val_accuracy: 0.7190 - val_loss: 0.5859
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 357s 3s/step - accuracy: 0.8395 - loss: 0.3726 - val_accuracy: 0.8430 - val_loss: 0.3645
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 351s 3s/step - accuracy: 0.8583 - loss: 0.3386 - val_accuracy: 0.8434 - val_loss: 0.3802
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 368s 3s/step - accuracy: 0.8750 - loss: 0.3064 - val_accuracy: 0.8637 - val_loss: 0.3298


In [44]:
lstm_model.evaluate(padded_test_sequences, y_test)

469/469 ━━━━━━━━━━━━━━━━━━━━ 73s 155ms/step - accuracy: 0.8682 - loss: 0.3225


[0.32245922088623047, 0.8682000041007996]

## Gated Recurrent Unit RNN

In [43]:
gru_model = Sequential()
gru_model.add(Embedding(input_dim=vocabulary_size, output_dim=128, input_length=maxlen))
gru_model.add(GRU(64, activation='tanh'))
gru_model.add(Dense(1, activation='sigmoid'))
gru_model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
gru_model.fit(padded_train_sequences, y_train, validation_split=.2, epochs=5, batch_size=256)

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 377s 3s/step - accuracy: 0.6338 - loss: 0.6300 - val_accuracy: 0.7263 - val_loss: 0.5304
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 394s 4s/step - accuracy: 0.7815 - loss: 0.4773 - val_accuracy: 0.7917 - val_loss: 0.4472
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 417s 4s/step - accuracy: 0.8226 - loss: 0.4029 - val_accuracy: 0.8246 - val_loss: 0.3907
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 399s 4s/step - accuracy: 0.8413 - loss: 0.3695 - val_accuracy: 0.8123 - val_loss: 0.4096
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 419s 4s/step - accuracy: 0.8558 - loss: 0.3423 - val_accuracy: 0.8486 - val_loss: 0.3649


In [45]:
gru_model.evaluate(padded_test_sequences, y_test)

469/469 ━━━━━━━━━━━━━━━━━━━━ 679s 1s/step - accuracy: 0.8492 - loss: 0.3624


[0.3623517155647278, 0.8492000102996826]

## Bidirectionl LSTM RNN

Allows to calculate the output for a given time ```t``` not only taking into account ```t-1``` but also ```t+1```.

In [47]:
bilstm_model = Sequential()
bilstm_model.add(Embedding(input_dim=vocabulary_size, output_dim=128, input_length=maxlen))
bilstm_model.add(Bidirectional(LSTM(64, activation='tanh')))
bilstm_model.add(Dense(1, activation='sigmoid'))
bilstm_model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
bilstm_model.fit(padded_train_sequences, y_train, validation_split=.2, epochs=5, batch_size=256)

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 6755s 62s/step - accuracy: 0.6229 - loss: 0.6408 - val_accuracy: 0.7220 - val_loss: 0.5374
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 6340s 58s/step - accuracy: 0.7864 - loss: 0.4605 - val_accuracy: 0.8307 - val_loss: 0.3887
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 6056s 56s/step - accuracy: 0.8316 - loss: 0.3888 - val_accuracy: 0.7913 - val_loss: 0.4439
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 3535s 32s/step - accuracy: 0.8508 - loss: 0.3545 - val_accuracy: 0.8614 - val_loss: 0.3318
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 536s 5s/step - accuracy: 0.8686 - loss: 0.3174 - val_accuracy: 0.8784 - val_loss: 0.2984


In [48]:
bilstm_model.evaluate(padded_test_sequences, y_test)

469/469 ━━━━━━━━━━━━━━━━━━━━ 82s 174ms/step - accuracy: 0.8807 - loss: 0.2915


[0.2915044128894806, 0.8807333111763]